In [1]:
import numpy as np
import pandas as pd
import torch

# NOTES: Compared to the source code chosen, there's gonna be differences that I haven't added into this notebook but the general ideas behind what I made is the same: i.e. the biochemical feature vectors + radius-based sequencing

In [2]:
AA_IDS = [
    'A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y','VOID',
]

# molecular weights are in kilodaltons
MOLECULAR_WEIGHTS = {
    'A': 0.09,
    'C': 0.12,
    'D': 0.13,
    'E': 0.15,
    'F': 0.17,
    'G': 0.08,
    'H': 0.16,
    'I': 0.13,
    'K': 0.15,
    'L': 0.13,
    'M': 0.15,
    'N': 0.13,
    'P': 0.12,
    'Q': 0.15,
    'R': 0.17,
    'S': 0.11,
    'T': 0.12,
    'V': 0.12,
    'W': 0.20,
    'Y': 0.18,
    'VOID': 0,
}

# net charges at physiological pH ~7.4
NET_CHARGES = {
    'A': 0,
    'C': 0,
    'D': -1,
    'E': -1,
    'F': 0,
    'G': 0,
    'H': 0,
    'I': 0,
    'K': +1,
    'L': 0,
    'M': 0,
    'N': 0,
    'P': 0,
    'Q': 0,
    'R': +1,
    'S': 0,
    'T': 0,
    'V': 0,
    'W': 0,
    'Y': 0,
    'VOID': 0,
}

# isoelectric point pHs from peptide web
# NOTE: will retrain model since I was using old textbook values with one outdated value -> shouldn't affect model accuracy much
ISOELECTRIC_PTS = {
    'A': 6.02,
    'C': 5.02,
    'D': 2.98,
    'E': 3.22,
    'F': 5.48,
    'G': 5.97,
    'H': 7.59,
    'I': 5.98,
    'K': 9.47,
    'L': 5.98,
    'M': 5.75,
    'N': 5.41,
    'P': 6.30,
    'Q': 5.65,
    'R': 10.76,
    'S': 5.68,
    'T': 5.60,
    'V': 5.97,
    'W': 5.94,
    'Y': 5.66,
    'VOID': 0,
}

# hydrophobicity levels - pH 7
# +ve values and up = hydrophobic
# around 0 = neutral
# -ve values = hydrophilic
HYDROPHOBICITY_IDXS = {
    'A': 41,
    'C': 49,
    'D': -55,
    'E': -31,
    'F': 100,
    'G': 0,
    'H': 8,
    'I': 100,
    'K': -23,
    'L': 97,
    'M': 74,
    'N': -28,
    'P': -46,
    'Q': -10,
    'R': -14,
    'S': -5,
    'T': 13,
    'V': 76,
    'W': 97,
    'Y': 63,
    'VOID': 0,
}


# individual half lives
HALF_LIFE = {
    'A': 4.4,
    'C': 1.2,
    'D': 1.1,
    'E': 1,
    'F': 1,
    'G': 30,
    'H': 3.5,
    'I': 20,
    'K': 1.3,
    'L': 5.5,
    'M': 30,
    'N': 1.4,
    'P': 21,
    'Q': 0.8,
    'R': 1,
    'S': 1.9,
    'T': 7.2,
    'V': 100,
    'W': 2.8,
    'Y': 2.8,
    'VOID': 0,
}


# pseudo molecular fingerprint accounting for biochemical / functional group standout features
# check function_fp_scheme.txt for more understanding
FUNCTIONAL_FP = {
    "A": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "G": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "D": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "E": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "K": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "R": [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "H": [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "N": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "Q": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "C": [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    "M": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
    "F": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "Y": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
    "W": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "L": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "I": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "V": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "S": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "T": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    "P": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    "VOID": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}

In [3]:
# do a bit of data exploration
import ast
import re
def parse_3d_col(s):
    arrays = re.findall(r"array\(\[(.*?)\]\)", s)
    return [np.array([float(x) for x in a.split(',')]) for a in arrays]
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
AA3D_df["AA_IDs"] = AA3D_df['AA_IDs'].apply(ast.literal_eval)
AA3D_df["3d_struct"] = AA3D_df["3d_struct"].apply(parse_3d_col)

In [4]:
all_vectors = np.concatenate(AA3D_df["3d_struct"].values)  # shape: (total_atoms, 3)
norms = np.linalg.norm(all_vectors, axis=1)

print(f"max radius: {norms.max():.4f} Å")
print(f"min radius: {norms.min():.4f} Å")

# max and min raw A directions were 25 and -27

max radius: 28.4715 Å
min radius: 0.5505 Å


In [5]:
class NAAnoEnc:
    """Run this everytime we need a new set of embeddings"""
    def __init__(self, verbose=False):
        self.nAAno_emb = {}   # basically just make it easier to embed down the line
        self.verbose = verbose

    def initialize(self):
        self.nAAno_emb = {aa_id: get_embedding(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEnc initialized")
        return True

    def get_aa_id(self, emb_vector):
        aa_id = None
        for code, key in self.nAAno_emb.items():
            if key == emb_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Embedding vector {emb_vector} presented does not exist")

        return aa_id


def get_embedding(aa_id: str):   # this isn't an embedding this is a feature extraction tool but I'm just gonna leave it as a learning point for myself
    """
    use this in these two scenarios:
    \n    - generating embeddings after updating feature vector
    \n    - inference
    :param aa_id: single letter amino acid reference code
    :returns: feature vector representing a given (valid) amino acid
    """
    # sanity check
    if aa_id not in AA_IDS:
        raise ValueError(f"{aa_id} not in valid AA ID list")

    # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
    embedding = [
        MOLECULAR_WEIGHTS[aa_id],
        NET_CHARGES[aa_id],
        ISOELECTRIC_PTS[aa_id],
        HYDROPHOBICITY_IDXS[aa_id],
        HALF_LIFE[aa_id],
    ]
    embedding += FUNCTIONAL_FP[aa_id]
    return embedding


def encoder_check():
    encoder = NAAnoEnc(verbose=True)
    encoder.initialize()
    for aa_code, aa_vect in encoder.nAAno_emb.items():
        print(f"{aa_code} -- {aa_vect}")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = get_embedding(aa)
        if aa_str == encoder.get_aa_id(aa_emb):
            print(f"{aa_str}: str <-> vect aligned")

encoder_check()  # note: all good

NAAnoEnc initialized
A -- [0.09, 0, 6.02, 41, 4.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C -- [0.12, 0, 5.02, 49, 1.2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]
D -- [0.13, -1, 2.98, -55, 1.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
E -- [0.15, -1, 3.22, -31, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
F -- [0.17, 0, 5.48, 100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
G -- [0.08, 0, 5.97, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H -- [0.16, 0, 7.59, 8, 3.5, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
I -- [0.13, 0, 5.98, 100, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
K -- [0.15, 1, 9.47, -23, 1.3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
L -- [0.13, 0, 5.98, 97, 5.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
M -- [0.15, 0, 5.75, 74, 30, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
N -- [0.13, 0, 5.41, -28, 1.4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
P -- [0.12, 0, 6.3, -46, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Q -- [0.15, 0, 5.65, -10, 0.8, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
R -- [0.17, 1,

In [6]:
import numpy as np
import torch

class RadialSeeker:

    def __init__(self, radial_resolution, intrashell_resolution, max_angstroms,
                 verbose=False):
        self.radial_resolution = radial_resolution
        self.angstrom_lim = max_angstroms   # maximum angstrom range found + some extra for context enrichment
                                 # can always edit this later on
        self.angstrom_inc = float(max_angstroms / radial_resolution)
        self.threshold = self.angstrom_inc / 2 # standardize how we determine radial sequences
        # getting half the angstrom increment will let us finely separate them

        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_lim, self.angstrom_inc, step= -1 * self.angstrom_inc)

        self.intrashell_resolution = intrashell_resolution
        self.intrashell_inc = max_angstroms / intrashell_resolution

        self.verbose = verbose


    def radial_sequence(self, aa_seq, vect_seq):
        # first order them by absolute distance to centroid
        # then convert them to indices according to intra-shell resolution

        # radial resolution determines how finely we want to order them
        # shell resolution determines how finely in 3D space we represent them

        radial_seq = []  # lookup table
        seen = set()

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its (radius, pos index) is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels: # go through all possible radius levels
            for i in range(len(aa_seq)):  # go through all amino acids in that sequence
                num_id=i
                if num_id not in seen and self._dist_check(np.linalg.norm(vect_seq[i]), level):
                    idx_vect = self.vect2idx(vect_seq[i])
                    radial_seq.append([[aa_seq[i]], idx_vect])
                    seen.add(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return radial_seq + [[['VOID'],[0, 0, 0]]]  # we want to go from outside inward

    def _dist_check(self, dist, ang_radius):
        if abs(dist - ang_radius) <= self.threshold:
            return True
        else:
            return False

    def vect2idx(self, vector):
        """
        ENCODE: Converts a torch.tensor of 3D vectors into their nearest shell-resolution's index
        """
        idxs = np.round(vector / self.intrashell_inc)
        idxs = np.clip(idxs, -self.intrashell_resolution, self.intrashell_resolution)
        return idxs.astype(int)

    def num2vect(self, idxs):
        """DECODE: Converts a torch.tensor of indexes into their max_Angstrom-scaled 3D vectors"""
        vector = []
        for number in idxs:
            vector.append(float(number) * self.angstrom_inc)
        return vector



def test_resolution():
    module = RadialSeeker(radial_resolution = 100, intrashell_resolution = 100, max_angstroms = 33)
    for level in module.radius_levels:
        print(level)
# test_resolution()  # all good

tensor(33.)
tensor(32.6700)
tensor(32.3400)
tensor(32.0100)
tensor(31.6800)
tensor(31.3500)
tensor(31.0200)
tensor(30.6900)
tensor(30.3600)
tensor(30.0300)
tensor(29.7000)
tensor(29.3700)
tensor(29.0400)
tensor(28.7100)
tensor(28.3800)
tensor(28.0500)
tensor(27.7200)
tensor(27.3900)
tensor(27.0600)
tensor(26.7300)
tensor(26.4000)
tensor(26.0700)
tensor(25.7400)
tensor(25.4100)
tensor(25.0800)
tensor(24.7500)
tensor(24.4200)
tensor(24.0900)
tensor(23.7600)
tensor(23.4300)
tensor(23.1000)
tensor(22.7700)
tensor(22.4400)
tensor(22.1100)
tensor(21.7800)
tensor(21.4500)
tensor(21.1200)
tensor(20.7900)
tensor(20.4600)
tensor(20.1300)
tensor(19.8000)
tensor(19.4700)
tensor(19.1400)
tensor(18.8100)
tensor(18.4800)
tensor(18.1500)
tensor(17.8200)
tensor(17.4900)
tensor(17.1600)
tensor(16.8300)
tensor(16.5000)
tensor(16.1700)
tensor(15.8400)
tensor(15.5100)
tensor(15.1800)
tensor(14.8500)
tensor(14.5200)
tensor(14.1900)
tensor(13.8600)
tensor(13.5300)
tensor(13.2000)
tensor(12.8700)
tensor(12.54

In [7]:
test = AA3D_df.iloc[1]
test_AAs = test["AA_IDs"]
test_3d = test["3d_struct"]

R = RadialSeeker(radial_resolution=100, intrashell_resolution=100, max_angstroms=33)
r_seq = R.radial_sequence(test_AAs, test_3d)
print("radial_order: ", r_seq)
print(len(test_AAs), len(test_3d))
print(len(r_seq))

radial_order:  [[['VOID'], [0, 0, 0]], [['I'], array([39, 57, 33])], [['G'], array([60, 65, 40])], [['C'], array([34, 56, 39])], [['Q'], array([29, 52, 51])], [['H'], array([53, 68, 43])], [['S'], array([52, 41, 34])], [['R'], array([46, 64, 38])], [['F'], array([35, 44, 41])], [['K'], array([33, 44, 51])], [['D'], array([37, 42, 60])], [['G'], array([32, 52, 46])], [['Y'], array([44, 59, 36])], [['V'], array([38, 59, 41])], [['P'], array([40, 39, 46])], [['E'], array([44, 38, 42])], [['S'], array([57, 61, 43])], [['L'], array([45, 44, 38])], [['V'], array([53, 59, 40])], [['Y'], array([41, 46, 59])], [['N'], array([50, 44, 38])], [['Y'], array([43, 55, 41])], [['Y'], array([40, 44, 49])], [['T'], array([48, 57, 43])], [['L'], array([43, 44, 54])], [['VOID'], [0, 0, 0]]]
24 24
26


In [8]:
from tqdm import tqdm

radial_module = RadialSeeker(radial_resolution=100, intrashell_resolution=100, max_angstroms=33)
result = []

for _, row in tqdm(AA3D_df.iterrows(), total=len(AA3D_df), desc="Generating radial sequences"):
    PDB_ID = row["PDB ID"]
    AA_seq = row["AA_IDs"]
    D3_Seq = row["3d_struct"]

    radial_sequence = radial_module.radial_sequence(AA_seq, D3_Seq)

    result_dict = {
        "PDB_ID": PDB_ID,
        "radial_sequence": radial_sequence
    }
    result.append(result_dict)

# NOTE: RADIAL SEQUENCES:
# for each row (radial layer) -> list of lists
# nested list: [0] is the AA_ID -> embed this with NAAnoEnc in training
#              [1] is the 3D vector: no embedding necessary

# I'm going to treat the protein space like a sequence, with wavenet
# Don't focus on the shell layers, just feed and embed the coordinates
# That way the model can still generate proteins on the same "shell"
# If the model samples a void token, end / break
# context would be like 10 positions, so the first aa would have 9 voids behind
# so total "vocab" size would be 100^3 so much more manageable damn its still a lot

# embedding table for the spatial tokens to sample from + predict where the next AA should be placed given its context and SMILES

Generating radial sequences: 100%|██████████| 19334/19334 [06:31<00:00, 49.39it/s]


In [9]:
result = pd.DataFrame(result)
result.to_csv("notebook_database/radial_seq_df.csv")
# this is the MAIN dataframe we'll use for everything now
# everything the model learns will be based around these radial sequences
# maybe should include the radius in it...

In [129]:
# all_pairs_df -> contains interactions for each amino acid vs each SMILES hit
#   we want our protein pockets to adapt to the drug structure
# molfp_df.csv -> all SMILES molecular fingerprints
# radial_seq_df.csv -> all protein pocket patch radial sequences